## BASE

In [49]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
import time
from tqdm.notebook import tqdm
from torch.utils.data import Subset
from collections import defaultdict
import random

In [50]:
DATA_ROOT = "/Users/alex/Developpement/Internship/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252

In [51]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [52]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE)

Using cache found in /Users/alex/.cache/torch/hub/facebookresearch_dinov2_main


In [53]:
def extract_features(loader, desc="Extracting features"):
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            all_feats.append(backbone(x.to(DEVICE)).cpu())
            all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       epochs=50, lr=1e-3, eval_every=10):
    head = nn.Linear(768, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Loaders sur features précalculées — tout en RAM, très rapide
    train_feat_loader = DataLoader(
        TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True
    )
    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    feats = feats.to(DEVICE)
                    preds = head(feats).argmax(dim=1).cpu()
                    all_preds.append(preds)
                    all_labels_val.append(y)

            all_preds      = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            f1_macro = f1_score(all_labels_val, all_preds, average="macro")
            f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | "
                f"Loss: {total_loss/total:.4f} | "
                f"Train Acc: {correct/total*100:.1f}% | "
                f"F1 Macro: {f1_macro*100:.1f}% | "
                f"F1 Weighted: {f1_weighted*100:.1f}%"
            )

    return head

In [ ]:
DISTILLED_PTH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_physics/data.pth"
distilled = torch.load(DISTILLED_PTH, map_location=DEVICE)
# distilled est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Distilled data keys: {distilled.keys()}")
images_d = distilled["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_d = distilled["labels"].to(DEVICE)
print(labels_d)
from torch.utils.data import TensorDataset
distill_loader = DataLoader(
    TensorDataset(images_d.cpu(), labels_d.cpu()),
    batch_size=20, shuffle=True
)

Distilled data keys: dict_keys(['syn_J', 'syn_T', 'syn_B', 'images', 'labels'])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])


Extracting features...


Test:   0%|          | 0/26 [00:00<?, ?it/s]

In [54]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds,   batch_size=64, shuffle=False, num_workers=4)

## Baselines



### Full data


In [ ]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True, num_workers=4)

In [56]:
print("Extracting features...")
test_feats,    test_labels    = extract_features(test_loader,    "Test")

Extracting features...


Test:   0%|          | 0/26 [00:00<?, ?it/s]

In [57]:
print("\nTraining on full data...")
start_time = time.time()
full_feats,    full_labels    = extract_features(full_loader,    "Full train")
head_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                epochs=50, eval_every=10)
print(f"Training time: {time.time() - start_time:.6f} seconds")


Training on full data...


Full train:   0%|          | 0/103 [00:00<?, ?it/s]

Training:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch  10/50 | Loss: 0.1425 | Train Acc: 95.3% | F1 Macro: 89.5% | F1 Weighted: 90.9%
Epoch  20/50 | Loss: 0.0966 | Train Acc: 96.8% | F1 Macro: 89.7% | F1 Weighted: 90.7%
Epoch  30/50 | Loss: 0.0751 | Train Acc: 97.7% | F1 Macro: 89.5% | F1 Weighted: 90.7%
Epoch  40/50 | Loss: 0.0622 | Train Acc: 98.3% | F1 Macro: 90.0% | F1 Weighted: 90.7%
Epoch  50/50 | Loss: 0.0518 | Train Acc: 98.7% | F1 Macro: 89.1% | F1 Weighted: 90.1%
Training time: 292.585717 seconds


### Random subset


In [58]:
ipc = len(images_d) // len(full_train.classes)
rng = random.Random(123)

class_to_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_train.samples):
    class_to_indices[label].append(idx)



stratified_indices = []
for label, indices in class_to_indices.items():
    stratified_indices.extend(rng.sample(indices, ipc))

random_subset = Subset(full_train, stratified_indices)

random_loader = DataLoader(
    random_subset,
    batch_size=len(stratified_indices),
    shuffle=True,
    num_workers=4,
)

In [59]:
print("\nTraining on random data...")
start_time = time.time()
random_feats, random_labels = extract_features(random_loader, "Distilled")
head_dist = train_linear_probe(random_feats, random_labels, test_feats, test_labels,
                                epochs=50, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")


Training on random data...


Distilled:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch  20/50 | Loss: 0.0007 | Train Acc: 100.0% | F1 Macro: 48.4% | F1 Weighted: 51.9%
Epoch  40/50 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 50.1% | F1 Weighted: 53.2%
Epoch  50/50 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 50.4% | F1 Weighted: 53.4%
Training time: 11.717349 seconds


### Random head

In [60]:
def evaluate_random_head(test_feats, test_labels, feat_dim=768):
    head = nn.Linear(feat_dim, NUM_CLASSES).to(DEVICE)
    head.eval()

    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    all_preds, all_labels_val = [], []
    with torch.no_grad():
        for feats, y in test_feat_loader:
            feats = feats.to(DEVICE)
            preds = head(feats).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels_val.append(y)

    all_preds      = torch.cat(all_preds).numpy()
    all_labels_val = torch.cat(all_labels_val).numpy()

    acc         = (all_preds == all_labels_val).mean()
    f1_macro    = f1_score(all_labels_val, all_preds, average="macro")
    f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

    print(
        f"Random head baseline | "
        f"Test Acc: {acc*100:.1f}% | "
        f"F1 Macro: {f1_macro*100:.1f}% | "
        f"F1 Weighted: {f1_weighted*100:.1f}%"
    )
    return head

print("\nBaseline: random (untrained) head...")
evaluate_random_head(test_feats, test_labels)


Baseline: random (untrained) head...
Random head baseline | Test Acc: 8.3% | F1 Macro: 5.3% | F1 Weighted: 11.1%


Linear(in_features=768, out_features=20, bias=True)

## Distilled data


### No Physics


In [61]:
NO_PHYSICS_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10/data.pth"
no_physics_data = torch.load(NO_PHYSICS_DATAPATH, map_location=DEVICE)
# no_physics_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"No physics data keys: {no_physics_data.keys()}")
images_no_physics = no_physics_data["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_no_physics = no_physics_data["labels"].to(DEVICE)
print(labels_no_physics)
from torch.utils.data import TensorDataset
no_physics_loader = DataLoader(
    TensorDataset(images_no_physics.cpu(), labels_no_physics.cpu()),
    batch_size=20, shuffle=True
)

No physics data keys: dict_keys(['images', 'labels'])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])


In [62]:
print("\nTraining on no physics data...")
start_time = time.time()
no_physics_feats, no_physics_labels = extract_features(no_physics_loader, "No Physics")
head_no_physics = train_linear_probe(no_physics_feats, no_physics_labels, test_feats, test_labels,
                                      epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")


Training on no physics data...


No Physics:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch  20/200 | Loss: 0.0014 | Train Acc: 100.0% | F1 Macro: 69.6% | F1 Weighted: 70.4%
Epoch  40/200 | Loss: 0.0003 | Train Acc: 100.0% | F1 Macro: 69.4% | F1 Weighted: 70.2%
Epoch  60/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 69.9% | F1 Weighted: 70.6%
Epoch  80/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 70.0% | F1 Weighted: 70.6%
Epoch 100/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 70.1% | F1 Weighted: 70.9%
Epoch 120/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 70.1% | F1 Weighted: 70.9%
Epoch 140/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 70.4% | F1 Weighted: 71.2%
Epoch 160/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 70.4% | F1 Weighted: 71.4%
Epoch 180/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 70.5% | F1 Weighted: 71.6%
Epoch 200/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 70.8% | F1 Weighted: 71.6%
Training time: 1.002127 seconds


### Physics

In [63]:
PHYSICS_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_physics/data.pth"
physics_data = torch.load(PHYSICS_DATAPATH, map_location=DEVICE)
# physics_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Physics data keys: {physics_data.keys()}")
images_physics = physics_data["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_physics = physics_data["labels"].to(DEVICE)
print(labels_physics)
from torch.utils.data import TensorDataset
physics_loader = DataLoader(
    TensorDataset(images_physics.cpu(), labels_physics.cpu()),
    batch_size=20, shuffle=True
)

Physics data keys: dict_keys(['syn_J', 'syn_T', 'syn_B', 'images', 'labels'])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])


In [64]:
print("\nTraining on physics data...")
start_time = time.time()
physics_feats, physics_labels = extract_features(physics_loader, "Physics")
head_physics = train_linear_probe(physics_feats, physics_labels, test_feats, test_labels,
                                   epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")


Training on physics data...


Physics:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch  20/200 | Loss: 0.0013 | Train Acc: 100.0% | F1 Macro: 66.6% | F1 Weighted: 73.1%
Epoch  40/200 | Loss: 0.0003 | Train Acc: 100.0% | F1 Macro: 67.0% | F1 Weighted: 73.5%
Epoch  60/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 67.0% | F1 Weighted: 73.4%
Epoch  80/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 67.1% | F1 Weighted: 73.3%
Epoch 100/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 67.2% | F1 Weighted: 73.4%
Epoch 120/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 67.8% | F1 Weighted: 73.5%
Epoch 140/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 68.0% | F1 Weighted: 73.6%
Epoch 160/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 68.5% | F1 Weighted: 73.7%
Epoch 180/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 68.6% | F1 Weighted: 73.8%
Epoch 200/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 68.8% | F1 Weighted: 73.8%
Training time: 1.164560 seconds


### SeaThru

In [65]:
SEATHRU_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_seathru/data.pth"
seathru_data = torch.load(SEATHRU_DATAPATH, map_location=DEVICE)
# seathru_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Seathru data keys: {seathru_data.keys()}")
images_seathru = seathru_data["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_seathru = seathru_data["labels"].to(DEVICE)
print(labels_seathru)
from torch.utils.data import TensorDataset
seathru_loader = DataLoader(
    TensorDataset(images_seathru.cpu(), labels_seathru.cpu()),
    batch_size=20, shuffle=True
)

Seathru data keys: dict_keys(['syn_J', 'syn_T', 'syn_d', 'syn_beta', 'syn_B', 'images', 'labels'])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])


In [66]:
print("\nTraining on seathru data...")
start_time = time.time()
seathru_feats, seathru_labels = extract_features(seathru_loader, "Seathru")
head_seathru = train_linear_probe(seathru_feats, seathru_labels, test_feats, test_labels,
                                   epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")


Training on seathru data...


Seathru:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch  20/200 | Loss: 0.0012 | Train Acc: 100.0% | F1 Macro: 72.4% | F1 Weighted: 78.9%
Epoch  40/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 73.6% | F1 Weighted: 79.7%
Epoch  60/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 73.9% | F1 Weighted: 79.8%
Epoch  80/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 73.4% | F1 Weighted: 79.3%
Epoch 100/200 | Loss: 0.0002 | Train Acc: 100.0% | F1 Macro: 73.8% | F1 Weighted: 79.3%
Epoch 120/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 73.8% | F1 Weighted: 79.4%
Epoch 140/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 73.8% | F1 Weighted: 79.3%
Epoch 160/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 73.8% | F1 Weighted: 79.1%
Epoch 180/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 73.9% | F1 Weighted: 79.3%
Epoch 200/200 | Loss: 0.0001 | Train Acc: 100.0% | F1 Macro: 73.9% | F1 Weighted: 79.1%
Training time: 1.018667 seconds
